# Reproduction notebook: Equal Weight Error, Unequal Effect: Quantization Sensitivity at the Consuming Operator

This notebook rebuilds every number reported in the paper from the JSON record of each run and
compares it with the value printed in the paper. A check fails if the two differ by more than
the rounding of the printed value.

**What it does not do.** It does not rerun the models. The records were produced by the scripts
listed in the last section, which need a GPU. It also does not rebuild the Lean file; the
section on formal checks prints the build record and the command.

Run it from anywhere inside the repository. It needs only the Python standard library.

The paper is in IEEE format, which numbers sections and tables in Roman numerals (Section V-C, Table V). This notebook writes the same numbers in digits (Section 5.3, Table 5). Table S1, the agreement table, and the tracer table are in the supplementary material.

In [1]:
import glob, json, math, os, re, shutil, statistics, subprocess, sys, tempfile

root = os.path.abspath(".")
while not os.path.isdir(os.path.join(root, "experiments", "results_matched_bit")):
    parent = os.path.dirname(root)
    assert parent != root, "run this notebook inside the repository"
    root = parent
RM = os.path.join(root, "experiments", "results_matched_bit")
BM = os.path.join(root, "benchmarks")
PAPER = os.path.join(root, "paper", "tnnls_operator")

def rec(folder, name):
    with open(os.path.join(folder, name), "rb") as fh:
        return json.loads(fh.read())

CHECKS = []

def check(where, printed, got, tol):
    """Record one comparison: the paper prints `printed`, the record gives `got`."""
    ok = abs(float(got) - float(printed)) <= tol * (1 + 1e-9)
    CHECKS.append((where, printed, float(got), ok))
    return ok

def report(prefix):
    rows = [c for c in CHECKS if c[0].startswith(prefix)]
    bad = [c for c in rows if not c[3]]
    print(f"{prefix}: {len(rows) - len(bad)} of {len(rows)} checks pass")
    for where, printed, got, ok in bad:
        print("  FAIL", where, "paper", printed, "record", got)

print("repository root found; records:", len(glob.glob(os.path.join(RM, "*.json"))), "in results_matched_bit")

repository root found; records: 30 in results_matched_bit


## Table S1: decision-level agreement, with intervals from resampling whole sequences

Record: `experiments/results_matched_bit/agreement_repeats_<model>.json`. The intervals are the
record's own bootstrap over sequences (2,000 resamples).

In [2]:
T1 = {  # model: bits: (accuracy, dAcc, [lo, hi], r, c, 1-A, [lo, hi], D-D0, [lo, hi])
 "Qwen2.5-1.5B-Instruct": {"base": 0.462,
   "8": (0.462, 0.000, (-0.002, 0.001), 0.003, 0.004, 0.022, (0.018, 0.026), 0.008, (0.004, 0.012)),
   "4": (0.367, 0.095, (0.085, 0.106), 0.131, 0.036, 0.409, (0.388, 0.429), 0.395, (0.376, 0.413)),
   "3": (0.002, 0.459, (0.439, 0.481), 0.460, 0.001, 0.996, (0.995, 0.998), 0.982, (0.979, 0.985))},
 "Mistral-7B-v0.1": {"base": 0.537,
   "8": (0.538, 0.000, (-0.001, 0.000), 0.002, 0.002, 0.012, (0.010, 0.013), 0.005, (0.003, 0.007)),
   "4": (0.495, 0.043, (0.037, 0.048), 0.072, 0.029, 0.232, (0.224, 0.240), 0.226, (0.217, 0.234)),
   "3": (0.000, 0.537, (0.519, 0.556), 0.537, 0.000, 0.996, (0.996, 0.996), 0.990, (0.988, 0.991))},
 "Llama-3.2-3B": {"base": 0.496,
   "8": (0.496, 0.000, (-0.002, 0.002), 0.003, 0.003, 0.022, (0.019, 0.025), 0.010, (0.008, 0.013)),
   "4": (0.429, 0.067, (0.057, 0.078), 0.100, 0.033, 0.319, (0.305, 0.332), 0.307, (0.293, 0.320)),
   "3": (0.054, 0.442, (0.420, 0.463), 0.451, 0.009, 0.919, (0.910, 0.927), 0.907, (0.899, 0.915))},
}
SEQS = {"Qwen2.5-1.5B-Instruct": (24, 6120), "Mistral-7B-v0.1": (48, 12240), "Llama-3.2-3B": (32, 8160)}
D0 = {  # ranges over three seed pairs, in percent, at dither 0.02, 0.05, 0.1 of a step
 "Llama-3.2-3B": [(1.1, 1.3), (1.8, 2.0), (2.3, 2.6)],
 "Qwen2.5-1.5B-Instruct": [(1.1, 1.4), (1.8, 2.1), (2.6, 3.0)],
 "Mistral-7B-v0.1": [(0.6, 0.8), (0.9, 1.0), (1.3, 1.4)],
}
h = 0.0005
for m, rows in T1.items():
    d = rec(RM, f"agreement_repeats_{m}.json")
    w = f"Table S1 {m}"
    check(f"{w} sequences", SEQS[m][0], d["n_seqs"], 0)
    check(f"{w} positions", SEQS[m][1], d["n_positions"], 0)
    check(f"{w} bootstrap resamples", 2000, d["bootstrap"]["n"], 0)
    assert d["bootstrap"]["unit"] == "sequence"
    check(f"{w} base accuracy", rows["base"], d["quantized"]["8"]["acc_base"], h)
    for b in ["8", "4", "3"]:
        q, p = d["quantized"][b], rows[b]
        check(f"{w} {b}-bit accuracy", p[0], q["acc_quant"], h)
        check(f"{w} {b}-bit dAcc", p[1], q["acc_drop"], h + 1e-4)
        check(f"{w} {b}-bit dAcc lo", p[2][0], q["acc_drop_ci95"][0], h)
        check(f"{w} {b}-bit dAcc hi", p[2][1], q["acc_drop_ci95"][1], h)
        check(f"{w} {b}-bit r", p[3], q["regressions"], h)
        check(f"{w} {b}-bit c", p[4], q["recoveries"], h)
        check(f"{w} {b}-bit 1-A", p[5], q["disagreement"], h)
        check(f"{w} {b}-bit 1-A lo", p[6][0], q["disagreement_ci95"][0], h)
        check(f"{w} {b}-bit 1-A hi", p[6][1], q["disagreement_ci95"][1], h)
        check(f"{w} {b}-bit D-D0", p[7], q["d_minus_d0_ref"], h)
        check(f"{w} {b}-bit D-D0 lo", p[8][0], q["d_minus_d0_ref_ci95"][0], h)
        check(f"{w} {b}-bit D-D0 hi", p[8][1], q["d_minus_d0_ref_ci95"][1], h)
        # the identities the Lean file proves, on the recorded counts
        assert abs((q["regressions"] - q["recoveries"]) - q["acc_drop"]) < 1e-9
        assert abs(q["ca"] - (q["acc_base"] - q["regressions"])) < 1e-9
    for (lo, hi), amp in zip(D0[m], ["0.02", "0.05", "0.1"]):
        v = d["d0_by_amplitude"][amp]
        check(f"{w} D0 range low at dither {amp} (%)", lo, 100 * min(v), 0.05)
        check(f"{w} D0 range high at dither {amp} (%)", hi, 100 * max(v), 0.05)
q = rec(RM, "agreement_repeats_Llama-3.2-3B.json")["quantized"]["8"]
check("Table S1 text Llama 8-bit Correctness Agreement", 0.493, q["ca"], h)
check("Table S1 text Llama 8-bit regressions (%)", 0.33, 100 * q["regressions"], 0.005)
check("Table S1 text Llama 8-bit recoveries (%)", 0.33, 100 * q["recoveries"], 0.005)
report("Table S1")

Table S1: 141 of 141 checks pass


## The first records: matched-bit ratios, the wavelength probe, routing and decay

These come from the first set of records. The paper's two earlier scripts,
`scripts/derive_numbers.py` and `scripts/derive_ratios.py`, print them. The cell below runs both
scripts on copies of the records and compares their output with the paper.

In [3]:
tmp = tempfile.mkdtemp(prefix="tmlr_records_")
names = ([f"behavioral_{m}.json" for m in ["Qwen2.5-1.5B-Instruct", "Mistral-7B-v0.1", "Llama-3.2-3B"]]
         + [f"matched_bit_{m}.json" for m in ["Qwen2.5-1.5B-Instruct", "gemma-3-4b-it", "Mistral-7B-v0.1", "Llama-3.2-3B"]]
         + [f"k_wavelength_{m}.json" for m in ["Llama-3.2-3B", "Mistral-7B-v0.1"]]
         + ["validate_rope_aware_k_Llama-3.2-3B.json"])
for n in names:
    shutil.copy(os.path.join(RM, n), tmp)
for n in ["results_mixtral_routing.json", "results_olmoe_routing.json", "results_mamba_decay.json"]:
    shutil.copy(os.path.join(BM, n), tmp)
out = ""
for script in ["derive_numbers.py", "derive_ratios.py"]:
    r = subprocess.run([sys.executable, "-B", os.path.join(PAPER, "scripts", script)], cwd=tmp,
                       capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-2000:]
    out += ("== three-decimal ratios\n" if script == "derive_ratios.py" else "") + r.stdout
shutil.rmtree(tmp, ignore_errors=True)
print(out[-1400:])

def block(model):
    """The lines of `out` for one model, per section, keyed by the section header."""
    res, sec = {}, None
    for line in out.splitlines():
        if line.startswith("== "):
            sec = line
        elif sec is not None and not line.startswith(" "):
            cur = line.split(" ")[0]
            res.setdefault((sec, cur), [line])
        elif sec is not None and res:
            res[list(res)[-1]].append(line)
    return {k: "\n".join(v) for k, v in res.items() if model in k[1]}

def num(pattern, text):
    m = re.search(pattern, text)
    assert m, pattern
    return float(m.group(1))

# Table 1: functional KL ratio QK / VO, and the range of the relative Frobenius ratio
# Ratios and intervals: the second run, experiments/results_matched_bit/per_sequence/.
# Frobenius ranges and the per-projection figures in the text: the first run.
T2 = {  # model: (Frobenius range, {bits: (ratio, lo, hi)}, 4-bit VO/QK (value, lo, hi))
 "Qwen2.5-1.5B-Instruct": ("Qwen/Qwen2.5-1.5B-Instruct", (0.94, 1.00),
   {"8": (0.38, 0.36, 0.40), "4": (0.26, 0.25, 0.28), "3": (0.17, 0.17, 0.18), "2": (0.27, 0.26, 0.27)}, (3.78, 3.54, 4.03)),
 "gemma-3-4b-it": ("unsloth/gemma-3-4b-it", (1.03, 1.04),
   {"8": (0.88, 0.82, 0.94), "4": (0.42, 0.39, 0.45), "3": (0.41, 0.37, 0.45)}, (2.37, 2.20, 2.59)),
 "Mistral-7B-v0.1": ("mistralai/Mistral-7B-v0.1", (1.01, 1.13),
   {"8": (0.28, 0.26, 0.30), "4": (0.29, 0.27, 0.32), "3": (0.45, 0.42, 0.47), "2": (1.05, 1.02, 1.08)}, (3.41, 3.09, 3.77)),
 "Llama-3.2-3B": ("unsloth/Llama-3.2-3B", (1.00, 1.05),
   {"8": (0.39, 0.37, 0.42), "4": (0.36, 0.34, 0.38), "3": (2.78, 2.56, 3.01), "2": (1.13, 1.11, 1.15)}, (2.75, 2.60, 2.92)),
}
ratio_lines = out[out.index("== three-decimal ratios"):].splitlines()
PS = os.path.join(RM, "per_sequence")
for tag, (m, frange, kl, inv4) in T2.items():
    d2 = rec(PS, f"matched_bit_{tag}.json")
    iv = d2["intervals"]
    assert iv["unit"] == "sequence" and iv["paired"] and iv["resamples"] == 2000, iv
    line = [l for l in ratio_lines if l.startswith(m)][0]
    wf = []
    for b, (printed, lo, hi) in kl.items():
        v = iv["by_bits"][b]
        check(f"Table 1 {m} {b}-bit KL ratio", printed, v["qk_over_vo"], 0.005)
        check(f"Table 1 {m} {b}-bit KL ratio lo", lo, v["ci95"][0], 0.005)
        check(f"Table 1 {m} {b}-bit KL ratio hi", hi, v["ci95"][1], 0.005)
        seg = re.search(rf"b{b} wf=([\d.]+) ok=([\d.]+) inv=([\d.]+)", line)
        wf.append(float(seg.group(1)))
        first = float(seg.group(2))
        assert v["ci95"][0] <= first <= v["ci95"][1], (m, b, "first-run ratio outside the interval")
        if b in ("8", "4"):
            assert v["ci95"][1] < 1.0, (m, b, "interval reaches 1")
    v4 = iv["by_bits"]["4"]
    check(f"Table 1 text {m} 4-bit VO/QK", inv4[0], 1 / v4["qk_over_vo"], 0.005)
    check(f"Table 1 text {m} 4-bit VO/QK lo", inv4[1], 1 / v4["ci95"][1], 0.005)
    check(f"Table 1 text {m} 4-bit VO/QK hi", inv4[2], 1 / v4["ci95"][0], 0.005)
    check(f"Table 1 {m} Frobenius ratio, low end", frange[0], min(wf), 0.005)
    check(f"Table 1 {m} Frobenius ratio, high end", frange[1], max(wf), 0.005)

mb = out[out.index("== matched-bit"):out.index("== k-wavelength")]
mis = mb[mb.index("mistralai/Mistral-7B-v0.1"):]
b4 = [l for l in mis.splitlines() if l.strip().startswith("b4")][0]
for name, printed in [("Q", 0.034), ("K", 0.036), ("V", 0.075), ("O", 0.048)]:
    check(f"Table 1 text Mistral 4-bit flip rate {name}", printed, num(rf"flip .*?{name}=([\d.]+)", b4), 0.0005)
lla = mb[mb.index("unsloth/Llama-3.2-3B"):mb.index("mistralai/Mistral-7B-v0.1")]
b3 = [l for l in lla.splitlines() if l.strip().startswith("b3")][0]
for name, printed in [("Q", 0.089), ("K", 0.741), ("V", 0.125), ("O", 0.174)]:
    check(f"Section 5.2 Llama 3-bit KL {name}", printed, num(rf"kl .*?{name}=([\d.e+-]+)", b3), 0.0005)
for name, printed in [("Q", 0.22), ("K", 1.71), ("V", 0.31), ("O", 0.43)]:
    check(f"Section 5.2 Llama 3-bit movement per unit weight error {name}", printed,
          num(rf"amp .*?{name}=([\d.]+)", b3), 0.005)
qw = mb[mb.index("Qwen/Qwen2.5-1.5B-Instruct"):mb.index("unsloth/gemma-3-4b-it")]
b3q = [l for l in qw.splitlines() if l.strip().startswith("b3")][0]
check("Section 5.2 Qwen 3-bit KL O", 0.624, num(r"kl .*?O=([\d.e+-]+)", b3q), 0.0005)
check("Section 5.2 Qwen 3-bit KL V", 0.183, num(r"kl .*?V=([\d.e+-]+)", b3q), 0.0005)
report("Table 1"); report("Section 5.2")

# Table 3 and the protection replicated on other hardware
kw = out[out.index("== k-wavelength"):out.index("== rope-aware")]
T3 = {"unsloth/Llama-3.2-3B": (0.741, 0.905, 0.72, 87, 0.12, 0.173),
      "mistralai/Mistral-7B-v0.1": (0.030, 0.571, 0.79, 49, 0.22, 0.031)}
for m, p in T3.items():
    seg = kw[kw.index(m):]
    seg = "\n".join(seg.splitlines()[:2])
    check(f"Table 3 {m} joint KL", p[0], num(r"full_k_kl=([\d.]+)", seg), 0.0005)
    check(f"Table 3 {m} octile Spearman", p[1], num(r"bucket_spearman=([\d.]+)", seg), 0.0005)
    check(f"Table 3 {m} two longest octiles share", p[2], num(r"top2_share=([\d.]+)", seg), 0.005)
    check(f"Table 3 {m} KL removed (%)", p[3], num(r"\((\d+)% recovered\)", seg), 0.5)
    check(f"Table 3 {m} row weight error Spearman", p[4], num(r"row_spearman_relfro=([\d.]+)", seg), 0.005)
    check(f"Table 3 {m} sum of single-octile KLs", p[5], num(r"sum kl_only=([\d.]+)", seg), 0.0005)
rp = out[out.index("== rope-aware"):out.index("== routing")]
for name, pat, printed, tol in [("unprotected", r"unprot=([\d.]+) prot=", 0.7352, 5e-5),
                                ("protected", r" prot=([\d.]+) prot8", 0.0994, 5e-5),
                                ("protected at 8 bits", r"prot8=([\d.]+)", 0.0994, 5e-5),
                                ("recovery (%)", r"recovery=([\d.]+)%", 86.5, 0.05),
                                ("whole model 4-bit protected", r"full b4 prot=([\d.]+)", 0.4668, 5e-5),
                                ("whole model 4-bit unprotected", r"full b4 prot=[\d.]+ unprot=([\d.]+)", 0.4787, 5e-5),
                                ("whole model gain (%)", r"gain ([\d.]+)%", 2.5, 0.05)]:
    check(f"Table 3 text, separate implementation: {name}", printed, num(pat, rp), tol)
report("Table 3")

# Table 7: routing. Margins, gate flips and the noise sweep come from the rerun on logits,
# benchmarks/routing_margin_logits_<model>.json. The earlier routing run gives the decision
# counts, and its gate flip rates must be reproduced.
rt = out[out.index("== routing"):out.index("== mamba")]
T8 = {  # sweep: sigma / median margin -> (flips %, pair law %, low / high ratio)
 "mistralai/Mixtral-8x7B-Instruct-v0.1": dict(k=2, dec=131072, p50=0.485, p10=0.076, g4=(16.6, 10.8), g3=(34.0, 2.68),
    sweep={0.01: (0.39, 0.39, None), 0.1: (3.8, 3.6, None), 0.25: (9.3, 8.3, 1219), 0.5: (18.0, 14.7, 16.2),
           1.0: (34.3, 23.8, 3.3), 2.0: (58.5, 33.6, 1.5)}),
 "allenai/OLMoE-1B-7B-0924": dict(k=8, dec=262144, p50=0.060, p10=0.0086, g4=(91.8, 1.09), g3=(98.7, 1.01),
    sweep={0.01: (0.42, 0.42, None), 0.1: (4.1, 3.9, None), 0.25: (10.1, 8.8, 1080), 0.5: (19.2, 15.0, 20.7),
           1.0: (34.7, 23.3, 4.2), 2.0: (57.4, 32.2, 1.9)})}
RL = {}
for m, p in T8.items():
    seg = rt[rt.index(m):]
    seg = "\n".join(seg.splitlines()[:5])
    tag = m.split("/")[1]
    w = f"Table 7 {tag}"
    d = RL[m] = rec(BM, f"routing_margin_logits_{tag}.json")
    assert d["top_k"] == p["k"]
    check(f"{w} decisions", p["dec"], num(r"decisions=(\d+)", seg), 0)
    check(f"{w} decisions, rerun", p["dec"], d["n_decisions"], 0)
    pc = d["margins"][f"k{p['k']}"]["percentiles"]
    check(f"{w} median margin", p["p50"], pc["50"], 0.0005)
    check(f"{w} 10th percentile margin", p["p10"], pc["10"], 0.0005 if p["p10"] > 0.01 else 5e-5)
    for bits, key in [("4", "g4"), ("3", "g3")]:
        g = d["gate_weight_quant"][bits]
        check(f"{w} gate {bits}-bit flips (%)", p[key][0], 100 * g["overall_flip_rate"], 0.05)
        check(f"{w} gate {bits}-bit low/high", p[key][1], g["low_over_high_ratio"], 0.05 if p[key][1] > 5 else 0.005)
        earlier = num(rf"gate {bits}-bit: flip=([\d.]+)", seg)
        assert abs(earlier - g["overall_flip_rate"]) < 5e-4, (m, bits, "gate flips differ from the earlier run")
    lv = {l["sigma_over_median_margin"]: l for l in d["noise_sweep"][f"k{p['k']}"]["levels"]}
    assert d["noise_sweep"][f"k{p['k']}"]["draws"] == 4
    for sg, (flip, law, ratio) in p["sweep"].items():
        tol = 0.005 if sg == 0.01 else 0.05
        check(f"{w} noise {sg} flips (%)", flip, 100 * lv[sg]["overall_flip_rate"], tol)
        check(f"{w} noise {sg} pair law (%)", law, 100 * lv[sg]["pair_prediction"], tol)
        if ratio is None:
            assert lv[sg]["high_margin_flip_rate"] == 0.0, "the caption says no token above the median flips"
        else:
            check(f"{w} noise {sg} low/high", ratio, lv[sg]["low_over_high_ratio"], 0.5 if ratio > 100 else 0.05)
    ck = d["checks"]
    assert ck["prob_max_abs_diff"] < 1.1e-3, "the text says the probabilities match to 1.1e-3"
    assert ck["topk_agreement_min"] > (0.997 if "OLMoE" in m else 0.999), ck["topk_agreement_min"]
ol = RL["allenai/OLMoE-1B-7B-0924"]["margins"]["k8"]["per_layer_median"]
check("Table 7 text: OLMoE layer median margin, low", 0.03, min(ol), 0.005)
check("Table 7 text: OLMoE layer median margin, high", 0.10, max(ol), 0.005)
check("Table 7 text: ratio of the two median margins", 8, 0.485 / 0.060, 0.1)
report("Table 7"); report("Table 7 text")

mam = out[out.index("== mamba"):out.index("== sha256")]
check("Section 8 baseline perplexity", 11.65, num(r"base=([\d.]+)", mam), 0.005)
check("Section 8 log grid perplexity", 14.44, num(r"log_tau=([\d.]+)", mam), 0.005)
check("Section 8 linear grid perplexity, first implementation (1e10)", 1.0, num(r"linear=([\d.e+]+)", mam) / 1e10, 0.05)
check("Section 8 largest steady-state gain", 42690, num(r"max_gain=(\d+)", mam), 0.5)
check("Section 8 channels", 2359296, num(r"n_ch=(\d+)", mam), 0)

Instruct.json
  eebf3aa3d9f64203 k_wavelength_Llama-3.2-3B.json
  2ff57549d05332d1 k_wavelength_Mistral-7B-v0.1.json
  4be151b46eda4252 matched_bit_Llama-3.2-3B.json
  d48486fbea545564 matched_bit_Mistral-7B-v0.1.json
  e5f113d9357ee5fc matched_bit_Qwen2.5-1.5B-Instruct.json
  eecfeebe1b60f184 matched_bit_gemma-3-4b-it.json
  3681d4744aa27aa0 results_mamba_decay.json
  2040231c28c82484 results_mixtral_routing.json
  29bec9e157b886d9 results_olmoe_routing.json
  53c6dab6088490b4 validate_rope_aware_k_Llama-3.2-3B.json
== three-decimal ratios
unsloth/Llama-3.2-3B b8 wf=1.050 ok=0.401 inv=2.491 flips=0.003-0.006 | b4 wf=1.049 ok=0.363 inv=2.752 flips=0.048-0.091 | b3 wf=1.038 ok=2.774 inv=0.360 flips=0.147-0.374 | b2 wf=1.005 ok=1.129 inv=0.886 flips=0.739-0.922
mistralai/Mistral-7B-v0.1 b8 wf=1.125 ok=0.291 inv=3.435 flips=0.001-0.005 | b4 wf=1.117 ok=0.293 inv=3.408 flips=0.034-0.075 | b3 wf=1.091 ok=0.445 inv=2.248 flips=0.086-0.139 | b2 wf=1.007 ok=1.050 inv=0.953 flips=0.687-0.791
Qw

True

## Table 2: where the two orderings part (the llama.cpp factorial)

Record: `experiments/results_matched_bit/llamacpp_factorial_<model>.json`.

In [4]:
ARMS = ["kquant_mix", "kquant_pure", "chan_mix", "chan_equal"]
TF = {  # (rel. Frobenius, |d kurtosis|, functional KL), rows in the order of ARMS
 ("Llama-3.2-3B", "Q2_K"): [(1.97, 4.24, 0.91), (1.00, 1.23, 0.17), (2.27, 333, 31.6), (1.00, 1.25, 1.13)],
 ("Llama-3.2-3B", "Q3_K_M"): [(2.17, 1.94, 1.12), (1.01, 1.20, 0.23), (2.42, 3.62, 15.3), (1.04, 1.52, 2.77)],
 ("Mistral-7B-v0.1", "Q2_K"): [(2.72, 6.25, 1.18), (1.02, 1.92, 0.057), (3.13, 205, 67.3), (1.01, 1.47, 1.05)],
 ("Mistral-7B-v0.1", "Q3_K_M"): [(2.16, 2.94, 0.61), (1.01, 1.02, 0.17), (2.54, 4.08, 2.47), (1.09, 1.57, 0.445)],
}
def tol_of(x):
    s_ = repr(x)
    dec = len(s_.split(".")[1]) if "." in s_ else 0
    return 0.5 * 10 ** (-dec) + 1e-12
fr = {}
for (m, level), rows in TF.items():
    d = rec(RM, f"llamacpp_factorial_{m}.json")["results"][level]
    fr[(m, level)] = d
    for arm, p in zip(ARMS, rows):
        a = d["arms"][arm]
        w = f"Table 2 {m} {level} {arm}"
        check(f"{w} rel. Frobenius", p[0], a["weight_rel_fro_qk_over_vo"], 0.005)
        check(f"{w} |d kurtosis|", p[1], a["weight_d_kurtosis_qk_over_vo"], tol_of(p[1]))
        check(f"{w} functional KL", p[2], a["functional_qk_over_vo"], tol_of(p[2]))
# the shipped tensor types quoted in the text
assert fr[("Llama-3.2-3B", "Q2_K")]["types_mix"] == {"Q": ["Q2_K"], "K": ["Q2_K"], "V": ["Q3_K"], "O": ["Q3_K"]}
assert fr[("Mistral-7B-v0.1", "Q2_K")]["types_mix"] == {"Q": ["Q2_K"], "K": ["Q2_K"], "V": ["Q4_K"], "O": ["Q3_K"]}
for m in ["Llama-3.2-3B", "Mistral-7B-v0.1"]:
    t = fr[(m, "Q3_K_M")]["types_mix"]
    assert t["Q"] == ["Q3_K"] and t["K"] == ["Q3_K"] and t["O"] == ["Q4_K"] and set(t["V"]) == {"Q4_K", "Q5_K"}
# ranges quoted in the text
allr = lambda arm, key: [fr[k]["arms"][arm][key] for k in TF]
check("Table 2 text shipped mix Frobenius low", 2.0, min(allr("kquant_mix", "weight_rel_fro_qk_over_vo")), 0.05)
check("Table 2 text shipped mix Frobenius high", 2.7, max(allr("kquant_mix", "weight_rel_fro_qk_over_vo")), 0.05)
check("Table 2 text shipped mix kurtosis low", 1.9, min(allr("kquant_mix", "weight_d_kurtosis_qk_over_vo")), 0.05)
check("Table 2 text shipped mix kurtosis high", 6.3, max(allr("kquant_mix", "weight_d_kurtosis_qk_over_vo")), 0.05)
check("Table 2 text equal type Frobenius low", 1.00, min(allr("kquant_pure", "weight_rel_fro_qk_over_vo")), 0.005)
check("Table 2 text equal type Frobenius high", 1.02, max(allr("kquant_pure", "weight_rel_fro_qk_over_vo")), 0.005)
check("Table 2 text equal type kurtosis low", 1.0, min(allr("kquant_pure", "weight_d_kurtosis_qk_over_vo")), 0.05)
check("Table 2 text equal type kurtosis high", 1.9, max(allr("kquant_pure", "weight_d_kurtosis_qk_over_vo")), 0.05)
check("Table 2 text per-channel shipped bits Frobenius low", 2.3, min(allr("chan_mix", "weight_rel_fro_qk_over_vo")), 0.05)
check("Table 2 text per-channel shipped bits Frobenius high", 3.1, max(allr("chan_mix", "weight_rel_fro_qk_over_vo")), 0.05)
check("Table 2 text per-channel equal bits Frobenius low", 1.00, min(allr("chan_equal", "weight_rel_fro_qk_over_vo")), 0.005)
check("Table 2 text per-channel equal bits Frobenius high", 1.09, max(allr("chan_equal", "weight_rel_fro_qk_over_vo")), 0.005)
inv = [1 / v for v in allr("kquant_pure", "functional_qk_over_vo")]
check("Table 2 text VO over QK at equal type, low (4 times)", 4, min(inv), 0.5)
check("Table 2 text VO over QK at equal type, high (17 times)", 17, max(inv), 0.5)
check("Table 2 text shipped mix functional low", 0.61, min(allr("kquant_mix", "functional_qk_over_vo")), 0.005)
check("Table 2 text shipped mix functional high", 1.18, max(allr("kquant_mix", "functional_qk_over_vo")), 0.005)
kl2c = [fr[(m, "Q2_K")]["arms"]["chan_equal"][p]["functional"]["kl"] for m in ["Llama-3.2-3B", "Mistral-7B-v0.1"] for p in "QKVO"]
kl2k = [fr[(m, "Q2_K")]["arms"]["kquant_pure"][p]["functional"]["kl"] for m in ["Llama-3.2-3B", "Mistral-7B-v0.1"] for p in "QKVO"]
check("Table 2 text per-channel 2-bit KL low", 2.4, min(kl2c), 0.05)
check("Table 2 text per-channel 2-bit KL high", 5.5, max(kl2c), 0.05)
check("Table 2 text K-quant 2-bit KL low", 0.006, min(kl2k), 0.0005)
check("Table 2 text K-quant 2-bit KL high", 0.23, max(kl2k), 0.005)
report("Table 2")

Table 2: 68 of 68 checks pass


## Score-error decomposition (Section 5.1, "a partial test")

Record: `experiments/results_matched_bit/score_error_<model>.json`.

In [5]:
se = {m: rec(RM, f"score_error_{m}.json")["results"]
      for m in ["Qwen2.5-1.5B-Instruct", "gemma-3-4b-it", "Mistral-7B-v0.1", "Llama-3.2-3B"]}
col = lambda key, field: [se[m][key][field] for m in se]
check("Score error: row-constant share, query, 4 bits, low", 0.47, min(col("Q4", "const_share_layer_median")), 0.005)
check("Score error: row-constant share, query, 4 bits, high", 0.52, max(col("Q4", "const_share_layer_median")), 0.005)
check("Score error: row-constant share, key, 4 bits, low", 0.22, min(col("K4", "const_share_layer_median")), 0.005)
check("Score error: row-constant share, key, 4 bits, high", 0.33, max(col("K4", "const_share_layer_median")), 0.005)
c4 = col("Q4", "centered_rel") + col("K4", "centered_rel")
c3 = col("Q3", "centered_rel") + col("K3", "centered_rel")
check("Score error: centered error, 4 bits, low (%)", 6, 100 * min(c4), 0.5)
check("Score error: centered error, 4 bits, high (%)", 11, 100 * max(c4), 0.5)
check("Score error: centered error, 3 bits, low (%)", 15, 100 * min(c3), 0.5)
check("Score error: centered error, 3 bits, high (%)", 24, 100 * max(c3), 0.5)
t4 = col("Q4", "attention_tv") + col("K4", "attention_tv")
t3 = col("Q3", "attention_tv") + col("K3", "attention_tv")
check("Score error: attention total variation, 4 bits, low", 0.02, min(t4), 0.005)
check("Score error: attention total variation, 4 bits, high", 0.06, max(t4), 0.005)
check("Score error: attention total variation, 3 bits, low", 0.06, min(t3), 0.005)
check("Score error: attention total variation, 3 bits, high", 0.13, max(t3), 0.005)
for m in se:
    check(f"Score error: sequences, {m}", 8, rec(RM, f"score_error_{m}.json")["n_seqs"], 0)
report("Score error")

Score error: 16 of 16 checks pass


## Table 4: row protection at a matched budget, on held-out text

Record: `experiments/results_matched_bit/k_row_protection_<model>.json`.

In [6]:
TP = {  # condition: (KL, [lo, hi], removed %) per model; random conditions: (KL low, KL high, removed low, removed high)
 "Llama-3.2-3B": {"none": (0.781, (0.715, 0.837), None), "shortest": (0.749, (0.688, 0.812), 4),
                  "longest": (0.089, (0.083, 0.095), 89), "act_error": (0.031, (0.030, 0.033), 96),
                  "weight_error": (0.028, (0.027, 0.029), 96),
                  "random_rows": (0.491, 0.732, 6, 37), "random_freqs": (0.350, 0.627, 20, 55)},
 "Mistral-7B-v0.1": {"none": (0.0279, (0.0263, 0.0297), None), "shortest": (0.0262, (0.0248, 0.0277), 6),
                     "longest": (0.0137, (0.0128, 0.0147), 51), "act_error": (0.0095, (0.0090, 0.0101), 66),
                     "weight_error": (0.0073, (0.0069, 0.0078), 74),
                     "random_rows": (0.0237, 0.0248, 11, 15), "random_freqs": (0.0238, 0.0251, 10, 15)},
}
pr = {}
for m, rows in TP.items():
    d = rec(RM, f"k_row_protection_{m}.json")
    pr[m] = d
    r = d["results"]
    t = 0.0005 if m.startswith("Llama") else 0.00005
    w = f"Table 4 {m}"
    assert d["eval_split"].endswith("validation") and d["calib_split"].endswith("train")
    check(f"{w} evaluation sequences", 64, d["n_eval_seqs"], 0)
    check(f"{w} calibration sequences", 16, d["n_calib_seqs"], 0)
    check(f"{w} bits per key weight", 3.625, d["bits_per_key_weight"], 0)
    assert d["protected_rows_per_kproj"] * 8 == d["rows_per_kproj"]
    for cond, p in rows.items():
        if cond.startswith("random"):
            ks = sorted(k for k in r if k.startswith(cond))
            assert len(ks) == 5
            kls = [r[k]["kl"] for k in ks]
            rem = [100 * r[k]["recovery_vs_none"] for k in ks]
            check(f"{w} {cond} KL low", p[0], min(kls), t)
            check(f"{w} {cond} KL high", p[1], max(kls), t)
            check(f"{w} {cond} removed low (%)", p[2], min(rem), 0.5)
            check(f"{w} {cond} removed high (%)", p[3], max(rem), 0.5)
        else:
            check(f"{w} {cond} KL", p[0], r[cond]["kl"], t)
            check(f"{w} {cond} KL lo", p[1][0], r[cond]["kl_ci95"][0], t)
            check(f"{w} {cond} KL hi", p[1][1], r[cond]["kl_ci95"][1], t)
            if p[2] is not None:
                check(f"{w} {cond} removed (%)", p[2], 100 * r[cond]["recovery_vs_none"], 0.5)
L, M = pr["Llama-3.2-3B"]["results"], pr["Mistral-7B-v0.1"]["results"]
lows = [L[k]["kl_minus_longest_ci95"][0] for k in L if k.startswith("random")]
check("Table 4 text: smallest lower end, random minus longest (above 0.22)", 0.22, min(lows), 0.01)
assert min(lows) > 0.22
check("Table 4 text: weight error minus longest, lo", -0.066, L["weight_error"]["kl_minus_longest_ci95"][0], 0.0005)
check("Table 4 text: weight error minus longest, hi", -0.056, L["weight_error"]["kl_minus_longest_ci95"][1], 0.0005)
check("Table 4 text: probe set, unprotected", 0.741, L["none"]["probe_set"]["kl"], 0.0005)
check("Table 4 text: probe set, longest", 0.099, L["longest"]["probe_set"]["kl"], 0.0005)
check("Table 4 text: unprotected damage ratio Llama / Mistral", 28, L["none"]["kl"] / M["none"]["kl"], 0.5)
check("Table 4 text: overlap of weight-error rows with longest octile, Llama (%)", 23, 100 * L["weight_error"]["overlap_with_longest"], 0.5)
check("Table 4 text: the same, Mistral (%)", 43, 100 * M["weight_error"]["overlap_with_longest"], 0.5)
report("Table 4")

Table 4: 68 of 68 checks pass


## Table 7, new rows: router error spread over margin

Record: `benchmarks/router_error_margin_<model>.json`. The record's field
`logit_recompute_rel_err` compares recomputed logits with the router module's output, which is a
softmax, so it validates nothing and is not used here. The flip rates below are compared with
the routing run instead.

In [7]:
RR = {"Mixtral-8x7B-Instruct-v0.1": {"4": ((0.40, 1.28, 8.2), 40.9, 16.6), "3": ((0.90, 2.9, 18.5), 12.5, 34.0)},
      "OLMoE-1B-7B-0924": {"4": ((7.5, 27.9, 191), 0.003, 91.8), "3": ((12.1, 48.4, 345), 0.0, 98.7)}}
rr = {}
for m, rows in RR.items():
    d = rec(BM, f"router_error_margin_{m}.json")
    rr[m] = d
    assert "per-tensor uniform grid over [min, max]" in d["quantizer"]
    for b, (q, below, flips) in rows.items():
        v = d["results"][b]
        w = f"Table 7 R {m} {b}-bit"
        for name, printed, got in zip(["10th percentile", "median", "90th percentile"], q,
                                      [v["R_p10"], v["R_median"], v["R_p90"]]):
            check(f"{w} {name}", printed, got, tol_of(printed))
        check(f"{w} tokens with R<1 (%)", below, 100 * v["frac_R_below_1"], 0.05 if below > 1 else 0.0005)
        check(f"{w} flips (%), against the routing run", flips, 100 * v["flip_rate"], 0.05)
        fb = [x["flip_rate_R_below_1"] for x in v["per_layer"] if x["flip_rate_R_below_1"] is not None]
        assert all(x == 0.0 for x in fb), "a token with R < 1 changed its route"
mx = rr["Mixtral-8x7B-Instruct-v0.1"]["results"]["4"]["per_layer"]
check("Table 7 text: Mixtral layers", 32, len(mx), 0)
check("Table 7 text: Mixtral layer share with R<1, low (%)", 27, 100 * min(x["frac_R_below_1"] for x in mx), 0.5)
check("Table 7 text: Mixtral layer share with R<1, high (%)", 73, 100 * max(x["frac_R_below_1"] for x in mx), 0.5)
report("Table 7 R"); report("Table 7 text")

Table 7 R: 20 of 20 checks pass


Table 7 text: 6 of 6 checks pass


## Section 8: what each decay grid does to the channels

Record: `benchmarks/results_mamba_decay_census.json`.

In [8]:
c = rec(BM, "results_mamba_decay_census.json")
g = c["grids"]
check("Section 8 census: channels", 2359296, c["n_channels"], 0)
check("Section 8 census: baseline perplexity", 11.65, c["ppl_baseline"], 0.005)
check("Section 8 census: smallest decay rate (1e-5)", 2, c["rate_abs_A"]["min"] / 1e-5, 0.5)
check("Section 8 census: largest decay rate (1e6)", 2.9, c["rate_abs_A"]["max"] / 1e6, 0.05)
check("Section 8 census: median decay rate", 9.6, c["rate_abs_A"]["median"], 0.05)
check("Section 8 census: linear step over median rate, typical layer", 383, c["linear_step_over_median_rate"]["median"], 0.5)
assert c["linear_step_over_median_rate"]["min"] >= 38
check("Section 8 census: linear grid top-level share (%)", 99.8, 100 * g["linear"]["top_level_fraction"], 0.05)
assert 100 * g["linear"]["top_level_fraction_by_layer"]["min"] >= 99.0
check("Section 8 census: linear grid, slower by 10x (%)", 94, 100 * g["linear"]["rate_ratio_all"]["frac_slower_10x"], 0.5)
check("Section 8 census: linear grid, median slowdown (about 200)", 200, 1 / g["linear"]["rate_ratio_all"]["median"], 15)
check("Section 8 census: linear grid perplexity, second implementation (1e9)", 5.9, g["linear"]["ppl"] / 1e9, 0.05)
assert g["linear"]["ppl"] > 1e9
check("Section 8 census: log grid perplexity", 14.44, g["log_tau"]["ppl"], 0.005)
check("Section 8 census: log grid median rate ratio", 0.99, g["log_tau"]["rate_ratio_all"]["median"], 0.005)
check("Section 8 census: log grid, slower by 10x (%)", 0, 100 * g["log_tau"]["rate_ratio_all"]["frac_slower_10x"], 0.05)
check("Section 8 census: log grid, changed by 2x either way (%)", 28, 100 * g["log_tau"]["rate_ratio_all"]["frac_changed_2x"], 0.5)
check("Section 8 census: per-row linear grid top-level share (%)", 43.5, 100 * g["linear_per_row"]["top_level_fraction"], 0.05)
check("Section 8 census: per-row linear grid, slower by 10x (%)", 9, 100 * g["linear_per_row"]["rate_ratio_all"]["frac_slower_10x"], 0.5)
check("Section 8 census: per-row linear grid perplexity (1e6)", 4.1, g["linear_per_row"]["ppl"] / 1e6, 0.05)
report("Section 8")

Section 8: 22 of 22 checks pass


## Section 8, second model: Mamba-370m

Record: `benchmarks/results_mamba_decay_census_mamba-370m.json`.

In [9]:
c2 = rec(BM, "results_mamba_decay_census_mamba-370m.json")
g2 = c2["grids"]
assert c2["model"] == "state-spaces/mamba-370m-hf" and c2["bits"] == 3
check("Section 8 second model: layers", 48, c2["n_layers"], 0)
check("Section 8 second model: channels", 1572864, c2["n_channels"], 0)
check("Section 8 second model: baseline perplexity", 13.99, c2["ppl_baseline"], 0.005)
check("Section 8 second model: log grid perplexity", 18.94, g2["log_tau"]["ppl"], 0.005)
check("Section 8 second model: linear grid top-level share (%)", 99.8, 100 * g2["linear"]["top_level_fraction"], 0.05)
check("Section 8 second model: linear grid, slower by 10x (%)", 96, 100 * g2["linear"]["rate_ratio_all"]["frac_slower_10x"], 0.5)
assert g2["linear"]["ppl"] > 1e11
check("Section 8 second model: per-row linear grid top-level share (%)", 51, 100 * g2["linear_per_row"]["top_level_fraction"], 0.5)
check("Section 8 second model: per-row linear grid perplexity (1e7)", 5.1, g2["linear_per_row"]["ppl"] / 1e7, 0.05)
check("Section 8 second model: log grid, slower by 10x (%)", 0, 100 * g2["log_tau"]["rate_ratio_all"]["frac_slower_10x"], 0.05)
check("Section 8 second model: log grid, changed by 2x (%)", 35, 100 * g2["log_tau"]["rate_ratio_all"]["frac_changed_2x"], 0.5)
check("Section 8 second model: log grid cost, perplexity ratio", 1.35, g2["log_tau"]["ppl"] / c2["ppl_baseline"], 0.005)
check("Section 8 second model: the same ratio on the first model", 1.24, g["log_tau"]["ppl"] / c["ppl_baseline"], 0.005)
report("Section 8 second model")

Section 8 second model: 12 of 12 checks pass


## Intervals on Llama-3.2-3B: the factorial, the score-error decomposition, the wavelength probe

Records: `experiments/results_matched_bit/per_sequence/` (commit 6ad3fd9). Each is a rerun of a
committed record with per-sequence values; this cell checks that the rerun reproduces the
committed point values, then the printed intervals.

In [10]:
PSQ = os.path.join(RM, "per_sequence")
f0, f1 = rec(RM, "llamacpp_factorial_Llama-3.2-3B.json"), rec(PSQ, "llamacpp_factorial_Llama-3.2-3B.json")
FI = {("Q2_K", "kquant_mix"): (0.91, 0.84, 1.02), ("Q2_K", "kquant_pure"): (0.17, 0.15, 0.19),
      ("Q2_K", "chan_mix"): (31.6, 29.9, 33.5), ("Q2_K", "chan_equal"): (1.13, 1.11, 1.15),
      ("Q3_K_M", "kquant_mix"): (1.12, 1.07, 1.17), ("Q3_K_M", "kquant_pure"): (0.23, 0.22, 0.24),
      ("Q3_K_M", "chan_mix"): (15.3, 14.0, 16.7), ("Q3_K_M", "chan_equal"): (2.77, 2.55, 3.01)}
for (lvl, arm), (pt, lo, hi) in FI.items():
    a = f1["results"][lvl]["arms"][arm]
    assert a["interval"] == {"unit": "sequence", "paired": True, "resamples": 2000, "seed": 0}
    old = [f0["results"][lvl]["arms"][arm][p]["functional"]["kl"] for p in "QKVO"]
    new = [a[p]["functional"]["kl"] for p in "QKVO"]
    assert max(abs(x / y - 1) for x, y in zip(old, new)) < 1e-6, (lvl, arm, "rerun differs from the first run")
    tol = 0.05 if pt > 5 else 0.005
    w = f"Table 2 interval {lvl} {arm}"
    check(w + " ratio", pt, a["functional_qk_over_vo"], 0.06 if pt > 5 else tol)
    check(w + " lo", lo, a["functional_qk_over_vo_ci95"][0], tol)
    check(w + " hi", hi, a["functional_qk_over_vo_ci95"][1], tol)
assert f1["results"]["Q2_K"]["arms"]["kquant_mix"]["functional_qk_over_vo_ci95"][1] > 1, "the text says it includes parity"
report("Table 2 interval")

s8, s32 = rec(PSQ, "score_error_Llama-3.2-3B.json"), rec(PSQ, "score_error_Llama-3.2-3B_n32.json")
s0 = rec(RM, "score_error_Llama-3.2-3B.json")
assert s8["n_seqs"] == 8 and s32["n_seqs"] == 32
for key, (pt, lo, hi, pt32) in {"Q4": (0.46, 0.45, 0.47, 0.46), "K4": (0.36, 0.34, 0.38, 0.35)}.items():
    r = s8["results"][key]
    assert abs(r["const_share"] / s0["results"][key]["const_share"] - 1) < 1e-6
    check(f"Score error interval {key} share", pt, r["const_share"], 0.005)
    check(f"Score error interval {key} lo", lo, r["intervals"]["const_share"][0], 0.005)
    check(f"Score error interval {key} hi", hi, r["intervals"]["const_share"][1], 0.005)
    check(f"Score error interval {key} share, 32 sequences", pt32, s32["results"][key]["const_share"], 0.005)
report("Score error interval")

w0, w1 = rec(RM, "k_wavelength_Llama-3.2-3B.json"), rec(PSQ, "k_wavelength_Llama-3.2-3B.json")
assert abs(w1["full_k"]["out_kl"] / w0["full_k"]["out_kl"] - 1) < 1e-6 and w1["n_seq"] == 32
iv = w1["intervals"]
assert iv["unit"] == "sequence" and iv["paired"] and iv["resamples"] == 2000
check("Table 3 interval: joint KL lo", 0.670, iv["full_k_kl"][0], 0.0005)
check("Table 3 interval: joint KL hi", 0.818, iv["full_k_kl"][1], 0.0005)
check("Table 3 interval: Spearman lo", 0.786, iv["bucket_spearman_kl_only"][0], 0.0005)
check("Table 3 interval: Spearman hi", 0.929, iv["bucket_spearman_kl_only"][1], 0.0005)
check("Table 3 interval: top-two share lo (%)", 71, 100 * iv["damage_share_top2_buckets"][0], 0.5)
check("Table 3 interval: top-two share hi (%)", 74, 100 * iv["damage_share_top2_buckets"][1], 0.5)
check("Table 3 interval: longest over shortest lo", 26, iv["kl_only_longest_over_shortest"][0], 0.5)
check("Table 3 interval: longest over shortest hi", 35, iv["kl_only_longest_over_shortest"][1], 0.5)
report("Table 3 interval")

Table 2 interval: 24 of 24 checks pass
Score error interval: 8 of 8 checks pass
Table 3 interval: 8 of 8 checks pass


## Section 5.3: a second quantizer (Llama-3.2-3B)

Records: `experiments/results_matched_bit/quantizer_control_projections_Llama-3.2-3B.json` and
`k_row_protection_mse_clip_Llama-3.2-3B.json`; the absmax row-protection record is the one behind
Table 4.

In [11]:
import numpy as np
qc = rec(RM, "quantizer_control_projections_Llama-3.2-3B.json")
assert qc["n_eval_seqs"] == 64 and "validation" in qc["eval_split"]
assert len(qc["quantizers"]["mse_clip"]["c_grid"]) == 31 and qc["quantizers"]["mse_clip"]["c_grid"][-1] == 1.0
QC = {  # (quantizer, bits): (Q, K, V, O, ratio, lo, hi, Frobenius low, high)
 ("absmax", "4"): (0.009, 0.011, 0.027, 0.026, 0.38, 0.36, 0.40, 0.18, 0.19),
 ("mse_clip", "4"): (0.009, 0.068, 0.048, 0.016, 1.20, 1.11, 1.29, 0.12, 0.12),
 ("absmax", "3"): (0.079, 0.781, 0.116, 0.158, 3.14, 2.92, 3.40, 0.40, 0.43),
 ("mse_clip", "3"): (0.052, 0.166, 0.121, 0.053, 1.25, 1.17, 1.33, 0.21, 0.22),
}
PJ = ["q_proj", "k_proj", "v_proj", "o_proj"]
for (qn, b), v in QC.items():
    r = qc["results"][qn][b]
    w = f"Table 5 {qn} {b}-bit"
    for p, printed in zip(PJ, v[:4]):
        check(f"{w} {p} KL", printed, r[p]["kl"], 0.0005)
    check(f"{w} ratio", v[4], r["qk_over_vo"], 0.005)
    check(f"{w} ratio lo", v[5], r["qk_over_vo_ci95"][0], 0.005)
    check(f"{w} ratio hi", v[6], r["qk_over_vo_ci95"][1], 0.005)
    fro = [r[p]["rel_fro_mean"] for p in PJ]
    check(f"{w} Frobenius low", v[7], min(fro), 0.005)
    check(f"{w} Frobenius high", v[8], max(fro), 0.005)
c4, c3 = qc["results"]["mse_clip"]["4"], qc["results"]["mse_clip"]["3"]
a4 = qc["results"]["absmax"]["4"]
check("Section 5.3 mean c at 4 bits", 0.6, np.mean([c4[p]["mean_clip_c"] for p in PJ]), 0.02)
check("Section 5.3 mean c at 3 bits", 0.45, np.mean([c3[p]["mean_clip_c"] for p in PJ]), 0.02)
check("Section 5.3 key KL, clipped over absmax at 4 bits", 6, c4["k_proj"]["kl"] / a4["k_proj"]["kl"], 0.1)
check("Section 5.3 key Frobenius, absmax 4 bits", 0.19, a4["k_proj"]["rel_fro_mean"], 0.005)
check("Section 5.3 key Frobenius, clipped 4 bits", 0.12, c4["k_proj"]["rel_fro_mean"], 0.005)
f4 = [c4[p]["rel_fro_mean"] for p in PJ]
check("Section 5.3 clipped 4-bit Frobenius, low", 0.117, min(f4), 0.0005)
check("Section 5.3 clipped 4-bit Frobenius, high", 0.125, max(f4), 0.0005)
k4 = [c4[p]["kl"] for p in PJ]
check("Section 5.3 clipped 4-bit KL spread", 8, max(k4) / min(k4), 0.3)
assert c4["o_proj"]["kl"] < a4["o_proj"]["kl"], "the text says the output projection's KL falls"
assert max(k4) == c4["k_proj"]["kl"] and max(c3[p]["kl"] for p in PJ) == c3["k_proj"]["kl"]
for b in ("4", "3"):  # the clipped scale lowers every weight error
    for p in PJ:
        assert qc["results"]["mse_clip"][b][p]["rel_fro_mean"] < qc["results"]["absmax"][b][p]["rel_fro_mean"]

kc = rec(RM, "k_row_protection_mse_clip_Llama-3.2-3B.json")["results"]
ka = rec(RM, "k_row_protection_Llama-3.2-3B.json")["results"]
assert rec(RM, "k_row_protection_mse_clip_Llama-3.2-3B.json")["bits_per_key_weight"] == 3.625
check("Section 5.3 rows: none KL", 0.166, kc["none"]["kl"], 0.0005)
check("Section 5.3 rows: none equals the key cell of Table 5", c3["k_proj"]["kl"], kc["none"]["kl"], 1e-6)
check("Section 5.3 rows: weight error KL", 0.050, kc["weight_error"]["kl"], 0.0005)
check("Section 5.3 rows: shortest KL", 0.197, kc["shortest"]["kl"], 0.0005)
for name, clip, absm in [("weight_error", 70, 96), ("act_error", 58, 96), ("longest", 29, 89), ("shortest", None, 4)]:
    if clip is not None:
        check(f"Section 5.3 rows: {name} recovery, clipped (%)", clip, 100 * kc[name]["recovery_vs_none"], 0.5)
    check(f"Section 5.3 rows: {name} recovery, absmax (%)", absm, 100 * ka[name]["recovery_vs_none"], 0.5)
assert kc["shortest"]["kl"] > kc["none"]["kl"]
for nm, lo, hi, lo_a, hi_a in [("random_rows", 9, 19, 6, 37)]:
    rr = [100 * kc[f"{k}_{s_}"]["recovery_vs_none"] for k in ("random_rows", "random_freqs") for s_ in range(5)]
    check("Section 5.3 rows: random recovery low, clipped (%)", lo, min(rr), 0.5)
    check("Section 5.3 rows: random recovery high, clipped (%)", hi, max(rr), 0.5)
    ra = [100 * ka[f"random_rows_{s_}"]["recovery_vs_none"] for s_ in range(5)]
    check("Section 5.3 rows: random recovery low, absmax (%)", lo_a, min(ra), 0.5)
    check("Section 5.3 rows: random recovery high, absmax (%)", hi_a, max(ra), 0.5)
report("Table 5"); report("Section 5.3")

Table 5: 36 of 36 checks pass
Section 5.3: 23 of 23 checks pass


## Section 5.3 on a second model (Qwen2.5-1.5B)

Records: `quantizer_control_projections_Qwen2.5-1.5B-Instruct.json` and
`k_row_protection_mse_clip_Qwen2.5-1.5B-Instruct.json`, from the same script.

In [12]:
qq = rec(RM, "quantizer_control_projections_Qwen2.5-1.5B-Instruct.json")
assert qq["n_eval_seqs"] == 64 and "validation" in qq["eval_split"]
QQ = {  # (quantizer, bits): (Q, K, V, O, ratio, lo, hi, Frobenius low, high)
 ("absmax", "4"): (0.010, 0.017, 0.038, 0.056, 0.28, 0.27, 0.29, 0.17, 0.18),
 ("mse_clip", "4"): (0.011, 0.029, 0.025, 0.027, 0.76, 0.73, 0.80, 0.12, 0.13),
 ("absmax", "3"): (0.053, 0.087, 0.173, 0.619, 0.18, 0.17, 0.18, 0.39, 0.41),
 ("mse_clip", "3"): (0.050, 0.119, 0.087, 0.131, 0.78, 0.75, 0.81, 0.22, 0.23),
}
for (qn, b), v in QQ.items():
    r = qq["results"][qn][b]
    w = f"Table 5 Qwen {qn} {b}-bit"
    for pj, printed in zip(PJ, v[:4]):
        check(f"{w} {pj} KL", printed, r[pj]["kl"], 0.0005)
    check(f"{w} ratio", v[4], r["qk_over_vo"], 0.005)
    check(f"{w} ratio lo", v[5], r["qk_over_vo_ci95"][0], 0.005)
    check(f"{w} ratio hi", v[6], r["qk_over_vo_ci95"][1], 0.005)
    fro = [r[pj]["rel_fro_mean"] for pj in PJ]
    check(f"{w} Frobenius low", v[7], min(fro), 0.005)
    check(f"{w} Frobenius high", v[8], max(fro), 0.005)
q4, q3, qa4, qa3 = (qq["results"][a][b] for a, b in [("mse_clip", "4"), ("mse_clip", "3"), ("absmax", "4"), ("absmax", "3")])
check("Section 5.3 Qwen key Frobenius, absmax 4 bits", 0.17, qa4["k_proj"]["rel_fro_mean"], 0.005)
check("Section 5.3 Qwen key Frobenius, clipped 4 bits", 0.12, q4["k_proj"]["rel_fro_mean"], 0.005)
f4 = [q4[pj]["rel_fro_mean"] for pj in PJ]
check("Section 5.3 Qwen clipped 4-bit Frobenius, low", 0.121, min(f4), 0.0005)
check("Section 5.3 Qwen clipped 4-bit Frobenius, high", 0.130, max(f4), 0.0005)
k4 = [q4[pj]["kl"] for pj in PJ]
check("Section 5.3 Qwen clipped 4-bit KL spread", 2.8, max(k4) / min(k4), 0.05)
kvo = [q4[pj]["kl"] for pj in ("k_proj", "v_proj", "o_proj")]
assert max(kvo) / min(kvo) < 1.15, "key, value and output within 15% of one another"
check("Section 5.3 Qwen query below the others, low", 2.4, min(kvo) / q4["q_proj"]["kl"], 0.05)
check("Section 5.3 Qwen query below the others, high", 2.8, max(kvo) / q4["q_proj"]["kl"], 0.05)
# the direction the two models share
for res in (qq["results"], qc["results"]):
    a, c = res["absmax"]["4"], res["mse_clip"]["4"]
    assert c["k_proj"]["kl"] > a["k_proj"]["kl"] and c["k_proj"]["rel_fro_mean"] < a["k_proj"]["rel_fro_mean"]
    assert c["o_proj"]["kl"] < a["o_proj"]["kl"]
    assert abs(c["qk_over_vo"] - 1) < abs(a["qk_over_vo"] - 1)
assert q4["qk_over_vo_ci95"][1] < 1, "no reversal on Qwen"
assert max(qa3[pj]["kl"] for pj in PJ) == qa3["o_proj"]["kl"], "Qwen's 3-bit failure under absmax is the output projection"
assert q3["o_proj"]["kl"] < qa3["o_proj"]["kl"]
for b in ("4", "3"):
    for pj in PJ:
        assert qq["results"]["mse_clip"][b][pj]["rel_fro_mean"] < qq["results"]["absmax"][b][pj]["rel_fro_mean"]
cq = [np.mean([qq["results"]["mse_clip"][b][pj]["mean_clip_c"] for pj in PJ]) for b in ("4", "3")]
check("Section 5.3 Qwen mean c at 4 bits", 0.6, cq[0], 0.05)
check("Section 5.3 Qwen mean c at 3 bits", 0.45, cq[1], 0.05)

kq = rec(RM, "k_row_protection_mse_clip_Qwen2.5-1.5B-Instruct.json")
assert kq["bits_per_key_weight"] == 3.625
kq = kq["results"]
check("Section 5.3 Qwen rows: none KL", 0.119, kq["none"]["kl"], 0.0005)
for name, pct in [("act_error", 81), ("weight_error", 67), ("longest", 40), ("shortest", -6)]:
    check(f"Section 5.3 Qwen rows: {name} recovery (%)", pct, 100 * kq[name]["recovery_vs_none"], 0.5)
rr = [100 * kq[f"{k}_{s_}"]["recovery_vs_none"] for k in ("random_rows", "random_freqs") for s_ in range(5)]
check("Section 5.3 Qwen rows: random recovery low (%)", -5, min(rr), 0.5)
check("Section 5.3 Qwen rows: random recovery high (%)", 22, max(rr), 0.5)
# Section 5.2: weight error ahead in three of four cases, activation error in the fourth;
# both ahead of the longest-wavelength rows in all four
km = rec(RM, "k_row_protection_Mistral-7B-v0.1.json")["results"]
cases = {"Llama absmax": ka, "Mistral absmax": km, "Llama clipped": kc, "Qwen clipped": kq}
ahead = {n: r["weight_error"]["kl"] < r["act_error"]["kl"] for n, r in cases.items()}
assert ahead == {"Llama absmax": True, "Mistral absmax": True, "Llama clipped": True, "Qwen clipped": False}, ahead
for n, r in cases.items():
    assert max(r["weight_error"]["kl"], r["act_error"]["kl"]) < r["longest"]["kl"], n
report("Table 5 Qwen"); report("Section 5.3 Qwen")

Table 5 Qwen: 36 of 36 checks pass
Section 5.3 Qwen: 16 of 16 checks pass


## Section 8, a linear grid on the time constant (both Mamba checkpoints)

Records: `benchmarks/results_mamba_decay_tau_grid_mamba-790m-hf.json` and `..._mamba-370m-hf.json`.

In [13]:
TAU = {  # tag: (baseline, per-tensor ppl, factor, lowest %, per-row ppl, rise %, lowest %, faster 10x %, slowest tenth changed 2x %, log-grid rise %)
 "790m": (11.65, 71.3, 6.1, 92, 13.60, 17, 46, 9, 0.4),
 "370m": (13.99, 162.5, 11.6, 95, 16.48, 18, 47, 13, 0.6),
}
for tag, (base, pt, fac, low, pr, rise, lowr, f10, s2) in TAU.items():
    d = rec(BM, f"results_mamba_decay_tau_grid_mamba-{tag}-hf.json")
    assert d["bits"] == 3 and d["ppl_window"] == 2048 and d["ppl_max_windows"] == 40
    a, b = d["grids"]["linear_tau"], d["grids"]["linear_tau_per_row"]
    w = f"Section 8 tau grid {tag}: "
    check(w + "baseline", base, d["ppl_baseline"], 0.005)
    check(w + "per-tensor perplexity", pt, a["ppl"], 0.05)
    check(w + "per-tensor factor", fac, a["ppl"] / d["ppl_baseline"], 0.05)
    check(w + "per-tensor lowest-level share (%)", low, 100 * a["lowest_tau_level_fraction"], 0.5)
    assert a["rate_ratio_all"]["frac_faster_10x"] > 0.9, "nearly all sped up tenfold"
    assert a["rate_ratio_all"]["frac_slower_2x"] == 0.0
    check(w + "per-row perplexity", pr, b["ppl"], 0.005)
    check(w + "per-row rise (%)", rise, 100 * (b["ppl"] / d["ppl_baseline"] - 1), 0.5)
    check(w + "per-row lowest-level share (%)", lowr, 100 * b["lowest_tau_level_fraction"], 0.5)
    check(w + "per-row faster by 10x (%)", f10, 100 * b["rate_ratio_all"]["frac_faster_10x"], 0.5)
    check(w + "per-row slowest tenth changed 2x (%)", s2, 100 * b["rate_ratio_slowest_tenth"]["frac_changed_2x"], 0.05)
    check(w + "entries per row", 16, d["n_channels"] / d["n_layers"] / {"790m": 3072, "370m": 2048}[tag], 0)
report("Section 8 tau grid")

Section 8 tau grid: 20 of 20 checks pass


## Section 8, a different family: RWKV-4

Record: `benchmarks/results_rwkv_decay_grid_rwkv-4-169m-pile.json`.

In [14]:
rw = rec(BM, "results_rwkv_decay_grid_rwkv-4-169m-pile.json")
gr = rw["grids"]
assert rw["model"] == "RWKV/rwkv-4-169m-pile" and rw["bits"] == 3
check("Section 8 RWKV: blocks", 12, rw["n_blocks"], 0)
check("Section 8 RWKV: channels", 9216, rw["n_channels"], 0)
check("Section 8 RWKV: smallest rate (1e-7)", 1.6, rw["decay_rate"]["min"] / 1e-7, 0.05)
check("Section 8 RWKV: largest rate", 27, rw["decay_rate"]["max"], 0.5)
check("Section 8 RWKV: median rate", 0.027, rw["decay_rate"]["median"], 0.0005)
check("Section 8 RWKV: windows", 20, rw["ppl_max_windows"], 0)
check("Section 8 RWKV: window length", 1024, rw["ppl_window"], 0)
check("Section 8 RWKV: baseline perplexity", 31.32, rw["ppl_baseline"], 0.005)
check("Section 8 RWKV: log grid perplexity", 32.32, gr["log"]["ppl"], 0.005)
check("Section 8 RWKV: log grid rise (%)", 3, 100 * (gr["log"]["ppl"] / rw["ppl_baseline"] - 1), 0.5)
check("Section 8 RWKV: rate grid perplexity", 46.83, gr["linear_rate"]["ppl"], 0.005)
check("Section 8 RWKV: rate grid rise (%)", 50, 100 * (gr["linear_rate"]["ppl"] / rw["ppl_baseline"] - 1), 0.5)
check("Section 8 RWKV: rate grid, typical block share on the level nearest zero (%)", 69,
      100 * gr["linear_rate"]["extreme_level_fraction_by_layer"]["median"], 0.5)
check("Section 8 RWKV: rate grid, slower by 10x (%)", 31, 100 * gr["linear_rate"]["rate_ratio_all"]["frac_slower_10x"], 0.5)
check("Section 8 RWKV: time-constant grid perplexity", 166.8, gr["linear_tau"]["ppl"], 0.05)
check("Section 8 RWKV: time-constant grid, faster by 10x (%)", 34, 100 * gr["linear_tau"]["rate_ratio_all"]["frac_faster_10x"], 0.5)
report("Section 8 RWKV")

Section 8 RWKV: 16 of 16 checks pass


## The linear law for selections (Sections 3 and 7)

The pair law is the average of Phi(-m / (sqrt(2) sigma)) over the measured margins. The rerun on
logits stores every decision's margin (`benchmarks/routing_margins_<model>.npz`), so this cell
recomputes the law from the margins and compares it with the record and with the measured flips.

In [15]:
LAW = {  # (model, boundary): (measured % at 0.01, pair law %, growth 0.01 -> 0.02, measured / law at 1x, at 2x)
 ("Mixtral-8x7B-Instruct-v0.1", "top"): (0.391, 0.390, 1.95, 1.44, 1.74),
 ("OLMoE-1B-7B-0924", "top"): (0.423, 0.419, 1.97, 1.49, 1.78),
 ("Mixtral-8x7B-Instruct-v0.1", "k1"): (0.325, 0.334, 2.06, None, None),
 ("OLMoE-1B-7B-0924", "k1"): (0.386, 0.384, 2.01, None, None),
}
import numpy as np
erfc = np.vectorize(math.erfc)
worst_small, flips_small = 0.0, {}
for (tag, which), (meas, law, growth, at1, at2) in LAW.items():
    d = rec(BM, f"routing_margin_logits_{tag}.json")
    key = f"k{d['top_k']}" if which == "top" else "k1"
    sw = d["noise_sweep"][key]
    lv = {l["sigma_over_median_margin"]: l for l in sw["levels"]}
    mg = np.load(os.path.join(BM, d["margins_file"]))["margin_" + key].astype(np.float64).ravel()
    assert mg.size == d["n_decisions"] and (mg > 0).all()
    assert abs(np.median(mg) - sw["median_margin"]) < 1e-6 * sw["median_margin"] + 1e-7
    w = f"Linear law {tag} {key}"
    for sg in (0.01, 1.0):  # Phi(-m / (sqrt2 sigma)) = erfc(m / (2 sigma)) / 2
        mine = float(np.mean(0.5 * erfc(mg / (2 * lv[sg]["sigma"]))))
        assert abs(mine / lv[sg]["pair_prediction"] - 1) < 2e-3, (tag, key, sg, mine, lv[sg]["pair_prediction"])
    check(f"{w} measured flips at 0.01 (%)", meas, 100 * lv[0.01]["overall_flip_rate"], 0.0005)
    check(f"{w} pair law at 0.01 (%)", law, 100 * lv[0.01]["pair_prediction"], 0.0005)
    check(f"{w} growth when the noise doubles", growth, lv[0.02]["overall_flip_rate"] / lv[0.01]["overall_flip_rate"], 0.005)
    assert 0.055 < lv[0.01]["sigma_over_p10_margin"] < 0.075, "the text says 0.06 to 0.07 of the 10th percentile"
    for sg in (0.01, 0.02, 0.05, 0.1):
        worst_small = max(worst_small, abs(lv[sg]["overall_flip_rate"] / lv[sg]["pair_prediction"] - 1))
    if which == "top":
        check(f"{w} measured / pair law at 1x", at1, lv[1.0]["overall_flip_rate"] / lv[1.0]["pair_prediction"], 0.005)
        check(f"{w} measured / pair law at 2x", at2, lv[2.0]["overall_flip_rate"] / lv[2.0]["pair_prediction"], 0.005)
        flips_small[tag] = lv[0.01]["overall_flip_rate"] * d["n_decisions"] * sw["draws"]
assert worst_small < 0.07, worst_small  # "within 7% of the pair law in all four cases"
check("Linear law: flips behind the Mixtral level at 0.01 (hundreds)", 20, flips_small["Mixtral-8x7B-Instruct-v0.1"] / 100, 0.6)
check("Linear law: flips behind the OLMoE level at 0.01 (hundreds)", 44, flips_small["OLMoE-1B-7B-0924"] / 100, 0.6)
report("Linear law")

Linear law: 18 of 18 checks pass


## Section 5.4 and Table 6: GPTQ in place of rounding

Records: `experiments/results_matched_bit/gptq_control_projections_{Llama-3.2-3B,Qwen2.5-1.5B-Instruct}.json`,
written by `experiments/k_gptq_control.py` (GPU). The rounding arms of these records must agree
with the records behind Table 5.

In [16]:
GQ = {  # model: arm: bits: (ratio, lo, hi)
 "Llama-3.2-3B": {
  "rtn_absmax": {"4": (0.38, 0.36, 0.40), "3": (3.14, 2.92, 3.40)},
  "gptq_absmax": {"4": (0.38, 0.36, 0.41), "3": (0.40, 0.39, 0.42)},
  "rtn_mse_clip": {"4": (1.20, 1.11, 1.29), "3": (1.25, 1.17, 1.32)},
  "gptq_mse_clip": {"4": (1.33, 1.23, 1.44), "3": (1.29, 1.20, 1.39)}},
 "Qwen2.5-1.5B-Instruct": {
  "rtn_absmax": {"4": (0.28, 0.27, 0.29), "3": (0.18, 0.17, 0.18)},
  "gptq_absmax": {"4": (0.42, 0.40, 0.44), "3": (0.44, 0.42, 0.46)},
  "rtn_mse_clip": {"4": (0.76, 0.72, 0.80), "3": (0.78, 0.75, 0.81)},
  "gptq_mse_clip": {"4": (1.25, 1.19, 1.31), "3": (1.58, 1.51, 1.67)}},
}
KL4 = {  # model: arm: KL of Q, K, V, O at 4 bits
 "Llama-3.2-3B": {"rtn_absmax": (.009, .011, .027, .026), "gptq_absmax": (.005, .005, .013, .012),
                  "rtn_mse_clip": (.009, .068, .048, .016), "gptq_mse_clip": (.004, .044, .030, .007)},
 "Qwen2.5-1.5B-Instruct": {"rtn_absmax": (.010, .017, .038, .056), "gptq_absmax": (.005, .008, .018, .012),
                           "rtn_mse_clip": (.011, .029, .025, .027), "gptq_mse_clip": (.004, .018, .011, .007)},
}
gp = {}
for m in GQ:
    d = rec(RM, f"gptq_control_projections_{m}.json")
    gp[m] = d["results"]
    assert d["n_eval_seqs"] == 64 and d["n_calib_seqs"] == 64 and "train" in d["calib_split"]
    assert d["gptq"] == {"blocksize": 128, "percdamp": 0.01, "sequential": True,
                         "row_scale": "fixed from the original weights, per grid", "arithmetic": "float64"}
    check(f"Section 5.4 {m}: calibration tokens (thousands)", 16.4, d["n_calib_tokens"] / 1000, 0.05)
    old = rec(RM, f"quantizer_control_projections_{m}.json")["results"]
    for arm, by_bits in GQ[m].items():
        for b, (val, lo, hi) in by_bits.items():
            r = gp[m][arm][b]
            w = f"Table 6 {m} {arm} {b}-bit"
            check(f"{w} ratio", val, r["qk_over_vo"], 0.005)
            check(f"{w} ratio lo", lo, r["qk_over_vo_ci95"][0], 0.005)
            check(f"{w} ratio hi", hi, r["qk_over_vo_ci95"][1], 0.005)
            if arm.startswith("rtn"):  # the same quantizer as the Table 5 record
                o = old[arm[4:]][b]
                for pj in PJ:
                    assert abs(r[pj]["kl"] - o[pj]["kl"]) < 2e-3 * max(o[pj]["kl"], 0.05), (m, arm, b, pj)
                assert abs(r["qk_over_vo"] - o["qk_over_vo"]) < 0.005
        for pj, printed in zip(PJ, KL4[m][arm]):
            check(f"Table 6 {m} {arm} 4-bit {pj} KL", printed, gp[m][arm]["4"][pj]["kl"], 0.0005)
    for b in ("4", "3"):  # the ordering of each row scale, and the key under the clipped scale
        assert gp[m]["gptq_absmax"][b]["qk_over_vo_ci95"][1] < 1 < gp[m]["gptq_mse_clip"][b]["qk_over_vo_ci95"][0]
        c = gp[m]["gptq_mse_clip"][b]
        assert max(c[pj]["kl"] for pj in PJ) == c["k_proj"]["kl"]
L, Q = gp["Llama-3.2-3B"], gp["Qwen2.5-1.5B-Instruct"]
g4 = L["gptq_absmax"]["4"]
red = [g4[pj]["output_err_rtn_same_inputs_mean"] / g4[pj]["output_err_mean"] for pj in PJ]
check("Section 5.4 output error, rounding over GPTQ, low", 3, min(red), 0.5)
check("Section 5.4 output error, rounding over GPTQ, high", 5, max(red), 0.5)
fro = [g4[pj]["rel_fro_mean"] for pj in PJ]
check("Section 5.4 GPTQ weight error low", 0.23, min(fro), 0.005)
check("Section 5.4 GPTQ weight error high", 0.26, max(fro), 0.005)
half = [g4[pj]["kl"] / L["rtn_absmax"]["4"][pj]["kl"] for pj in PJ]
assert 0.38 < min(half) and max(half) < 0.6, half
check("Section 5.4 Llama 3-bit key KL, rounding", 0.781, L["rtn_absmax"]["3"]["k_proj"]["kl"], 0.0005)
check("Section 5.4 Llama 3-bit key KL, GPTQ", 0.025, L["gptq_absmax"]["3"]["k_proj"]["kl"], 0.0005)
check("Section 5.4 Qwen 3-bit output KL, rounding", 0.619, Q["rtn_absmax"]["3"]["o_proj"]["kl"], 0.0005)
check("Section 5.4 Qwen 3-bit output KL, GPTQ", 0.062, Q["gptq_absmax"]["3"]["o_proj"]["kl"], 0.0005)
qo = Q["rtn_mse_clip"]["4"]["o_proj"]["kl"] / Q["gptq_mse_clip"]["4"]["o_proj"]["kl"]
qk = Q["rtn_mse_clip"]["4"]["k_proj"]["kl"] / Q["gptq_mse_clip"]["4"]["k_proj"]["kl"]
check("Section 5.4 Qwen output KL, rounding over GPTQ", 4, qo, 0.1)
check("Section 5.4 Qwen key KL, rounding over GPTQ", 1.6, qk, 0.05)
check("Section 5.4 Llama key output error, absmax", 0.0019, L["gptq_absmax"]["4"]["k_proj"]["output_err_mean"], 0.00005)
check("Section 5.4 Llama key output error, clipped", 0.0016, L["gptq_mse_clip"]["4"]["k_proj"]["output_err_mean"], 0.00005)
check("Section 5.4 Llama key KL, clipped over absmax", 9, L["gptq_mse_clip"]["4"]["k_proj"]["kl"] / L["gptq_absmax"]["4"]["k_proj"]["kl"], 0.5)
check("Section 5.4 Qwen key output error, absmax", 0.0056, Q["gptq_absmax"]["4"]["k_proj"]["output_err_mean"], 0.00005)
check("Section 5.4 Qwen key output error, clipped", 0.0048, Q["gptq_mse_clip"]["4"]["k_proj"]["output_err_mean"], 0.00005)
report("Table 6"); report("Section 5.4")

Table 6: 80 of 80 checks pass
Section 5.4: 17 of 17 checks pass


## Section 8: intervals over windows, and the log grid with a range per row

Records: `benchmarks/results_mamba_decay_intervals_mamba-{790m,370m}-hf.json`, written by
`benchmarks/mamba_decay_intervals.py`. Each holds the loss of every evaluation window, so the
cell recomputes the paired bootstrap as well as reading the stored intervals.

In [17]:
IV = {  # tag: arm: (ratio or rise in %, lo, hi, tolerance, as a rise in percent?)
 "790m": {"log_tau": (1.24, 1.23, 1.25, 0.005, False), "linear_tau": (6.1, 5.9, 6.4, 0.05, False),
          "linear_tau_per_row": (17, 16, 18, 0.5, True), "log_tau_per_row": (2.3, 2.1, 2.5, 0.05, True)},
 "370m": {"log_tau": (1.35, 1.34, 1.37, 0.005, False), "linear_tau": (11.6, 10.9, 12.4, 0.05, False),
          "linear_tau_per_row": (18, 17, 19, 0.5, True), "log_tau_per_row": (2.0, 1.8, 2.2, 0.05, True)},
}
PPL = {"790m": (11.65, {"log_tau": 14.44, "linear_tau": 71.3, "linear_tau_per_row": 13.60, "log_tau_per_row": 11.91}),
       "370m": (13.99, {"log_tau": 18.94, "linear_tau": 162.5, "linear_tau_per_row": 16.48, "log_tau_per_row": 14.27})}
CH2 = {"790m": 0.04, "370m": 0.13}
for tag in ("790m", "370m"):
    d = rec(BM, f"results_mamba_decay_intervals_mamba-{tag}-hf.json")
    assert d["bits"] == 3 and d["row_length"] == 16 and d["ppl_window"] == 2048
    bl, bt = np.array(d["baseline"]["window_loss"]), np.array(d["baseline"]["window_tokens"], dtype=float)
    check(f"Section 8 intervals {tag}: windows", 40, len(bl), 0)
    check(f"Section 8 intervals {tag}: baseline perplexity", PPL[tag][0], d["baseline"]["ppl"], 0.005)
    idx = np.random.default_rng(0).integers(0, len(bl), size=(2000, len(bl)))
    pb = np.exp((bl[idx] * bt[idx]).sum(1) / bt[idx].sum(1))
    for arm, (val, lo, hi, tol, rise) in IV[tag].items():
        a = d["arms"][arm]
        al = np.array(a["window_loss"])
        pa = np.exp((al[idx] * bt[idx]).sum(1) / bt[idx].sum(1))
        again = np.percentile(pa / pb, [2.5, 97.5])
        assert np.allclose(again, a["ratio_ci95"], rtol=1e-9), (tag, arm, again, a["ratio_ci95"])
        assert abs(np.exp((al * bt).sum() / bt.sum()) - a["ppl"]) < 1e-9
        f = (lambda x: 100 * (x - 1)) if rise else (lambda x: x)
        w = f"Section 8 intervals {tag} {arm}"
        check(f"{w}: perplexity", PPL[tag][1][arm], a["ppl"], 0.05 if a["ppl"] > 50 else 0.005)
        check(f"{w}: ratio", val, f(a["ratio_to_baseline"]), tol)
        check(f"{w}: ratio lo", lo, f(a["ratio_ci95"][0]), tol)
        check(f"{w}: ratio hi", hi, f(a["ratio_ci95"][1]), tol)
    check(f"Section 8 intervals {tag}: per-row log grid, changed by 2x (%)", CH2[tag],
          100 * d["arms"]["log_tau_per_row"]["rate_ratio_all"]["frac_changed_2x"], 0.005)
    # the per-row log grid is better than the per-row time-constant grid, intervals apart
    assert d["arms"]["log_tau_per_row"]["ratio_ci95"][1] < d["arms"]["linear_tau_per_row"]["ratio_ci95"][0]
report("Section 8 intervals")

Section 8 intervals: 38 of 38 checks pass


## Section 8: the grid uniform in the rate, with exact endpoints

Records: `benchmarks/results_mamba_decay_rate_safe_mamba-{790m,370m}-hf.json`, written by
`benchmarks/mamba_decay_rate_safe.py`. The top level is exactly the slowest rate and nothing is
clamped. The record also counts what the earlier `lo + idx * step` reconstruction clamps.

In [18]:
def mant(x):
    e = math.floor(math.log10(x))
    return round(x / 10**e, 1), e

SAFE = {  # tag: (per-tensor ppl, ratio, lo, hi, per-row ppl) as (mantissa, exponent)
 "790m": ((1.3, 10), (1.1, 9), (5.0, 8), (2.2, 9), None),
 "370m": ((3.8, 9), (2.7, 8), (1.4, 8), (5.0, 8), (5.1, 7)),
}
CL = {"790m": (392309, 17, 8, 12, 1, 99.8, 94), "370m": (359979, 23, 11, 25, 2, 99.8, 96)}
sf = {}
for tag in ("790m", "370m"):
    d = rec(BM, f"results_mamba_decay_rate_safe_mamba-{tag}-hf.json")
    sf[tag] = d
    assert d["bits"] == 3 and d["baseline_source"] == f"results_mamba_decay_intervals_mamba-{tag}-hf.json"
    t, r = d["arms"]["rate_safe"], d["arms"]["rate_safe_per_row"]
    assert t["levels_at_or_above_zero"] == 0 and r["levels_at_or_above_zero"] == 0
    for arm in (t, r):  # the 64-bit reconstruction clamps nothing either
        assert arm["earlier_lo_plus_idx_step"]["float64"] == {"channels_clamped": 0, "layers_with_a_clamp": 0}
    n, pct, layers, n_row, layers_row, top, slow = CL[tag]
    e32 = t["earlier_lo_plus_idx_step"]["float32"]
    check(f"Section 8 endpoints {tag}: channels the 32-bit grid clamps", n, e32["channels_clamped"], 0)
    check(f"Section 8 endpoints {tag}: share clamped (%)", pct, 100 * e32["channels_clamped"] / d["n_channels"], 0.5)
    check(f"Section 8 endpoints {tag}: layers with a clamp", layers, e32["layers_with_a_clamp"], 0)
    e32r = r["earlier_lo_plus_idx_step"]["float32"]
    check(f"Section 8 endpoints {tag}: per-row channels clamped", n_row, e32r["channels_clamped"], 0)
    check(f"Section 8 endpoints {tag}: per-row layers with a clamp", layers_row, e32r["layers_with_a_clamp"], 0)
    check(f"Section 8 endpoints {tag}: top-level share, exact endpoints (%)", top, 100 * t["frac_top_level"], 0.05)
    # the paper prints the share for the 32-bit grid; with exact endpoints it is within 0.6 points
    check(f"Section 8 endpoints {tag}: slowed by 10x, exact endpoints (%)", slow, 100 * t["frac_slowed_10x"], 0.6)
    pp, ra, lo, hi, row = SAFE[tag]
    for label, printed, got in [("perplexity", pp, t["ppl"]), ("ratio", ra, t["ratio_to_baseline"]),
                                ("ratio lo", lo, t["ratio_ci95"][0]), ("ratio hi", hi, t["ratio_ci95"][1])]:
        m_, e_ = mant(got)
        check(f"Section 8 endpoints {tag}: per-tensor {label}, mantissa", printed[0], m_, 0.05)
        check(f"Section 8 endpoints {tag}: per-tensor {label}, exponent", printed[1], e_, 0)
    assert t["ratio_ci95"][0] > 1e8, "a collapse on any reading"
    if row is None:  # this checkpoint's per-row pass had not finished when the paper was built
        assert "ppl" not in r, "fold the per-row result of Mamba-790m into the paper"
        continue
    m_, e_ = mant(r["ppl"])
    check(f"Section 8 endpoints {tag}: per-row perplexity, mantissa", row[0], m_, 0.05)
    check(f"Section 8 endpoints {tag}: per-row perplexity, exponent", row[1], e_, 0)
    assert r["ratio_ci95"][0] > 1e4
# the per-row grid was almost untouched by the clamp, and its perplexity agrees with the 32-bit run
for tag, census in [("790m", "results_mamba_decay_census.json"), ("370m", "results_mamba_decay_census_mamba-370m.json")]:
    old = rec(BM, census)["grids"]["linear_per_row"]["ppl"]
    new = sf[tag]["arms"]["rate_safe_per_row"].get("ppl")
    if new is None:
        continue
    assert abs(math.log10(new / old)) < 0.1, (tag, old, new)
report("Section 8 endpoints")

Section 8 endpoints: 32 of 32 checks pass


## The tracer on models with known consuming operators (Section 6; supplement, Sections S-II and S-III)

Record: `experiments/results_matched_bit/tracer_accuracy.json`, written by
`experiments/tracer_accuracy.py` (CPU, random weights, no checkpoint). Three modes: `structural`
reads names, `graph` reads the autograd graph of one forward pass and no name, `auto` uses both.

In [19]:
tr = rec(RM, "tracer_accuracy.json")
assert not tr["group_a_skipped"], tr["group_a_skipped"]
assert tr["transformers"] == "4.46.3"
GA = tr["group_a"]
EIGHT = {"Llama", "Mistral", "Qwen2", "Gemma", "GPT-2", "Mixtral", "OLMoE", "Mamba"}
assert set(GA["graph"]["per_model"]) == EIGHT | {"RWKV"}
for mode, right in [("structural", 200), ("graph", 200)]:
    P = GA[mode]["pooled"]
    check(f"Tracer table, eight architectures, {mode}: models", 8, P["models"], 0)
    check(f"Tracer table, eight architectures, {mode}: scored", 200, P["scored"], 0)
    check(f"Tracer table, eight architectures, {mode}: correct", right, P["correct"], 0)
check("Tracer table, eight architectures: symbolic trace succeeds", 0, GA["graph"]["pooled"]["models_where_symbolic_trace_ok"], 0)
check("Tracer table, eight architectures: forward-pass trace succeeds", 8, GA["graph"]["pooled"]["models_where_forward_pass_traced"], 0)
for mode, right in [("structural", 14), ("graph", 16), ("auto", 14)]:
    P = GA[mode]["pooled_blind"]
    check(f"Tracer table, RWKV, {mode}: scored", 16, P["scored"], 0)
    check(f"Tracer table, RWKV, {mode}: correct", right, P["correct"], 0)
check("Tracer table, RWKV: symbolic trace succeeds", 0, GA["graph"]["pooled_blind"]["models_where_symbolic_trace_ok"], 0)
check("Tracer table, RWKV: forward-pass trace succeeds", 1, GA["graph"]["pooled_blind"]["models_where_forward_pass_traced"], 0)
for v in GA["graph"]["per_model"].values():
    assert v["fx"]["symbolic_trace"] == "failed" and "input" in v["fx"]["error"], v["fx"]

# 30 of the 200 labels rest on graph evidence: 16 + 8 + 4 + 2
G = GA["graph"]["per_model"]
by = {m: G[m]["scored_by_method"].get("autograd", 0) for m in EIGHT}
check("Section 6: labels from graph evidence", 30, sum(by.values()), 0)
check("Section 6: default labels", 170, sum(G[m]["scored_by_method"].get("default", 0) for m in EIGHT), 0)
assert [by[m] for m in ("Llama", "Mistral", "Qwen2", "Gemma")] == [4, 4, 4, 4]
assert by["Mixtral"] == by["OLMoE"] == 6 and by["Mamba"] == 2 and by["GPT-2"] == 0
for m in ("Mixtral", "OLMoE"):
    lab = G[m]["labelled_by_graph"]
    assert sum(v == "gate_selection" for v in lab.values()) == 2 and sum(v == "softmax_score" for v in lab.values()) == 4
    g_ = G[m]["accuracy_by_true_regime"]["gate_selection"]
    assert g_["correct"] == g_["n"] == 2
assert all(not G[m]["wrong"] for m in G)
# the RWKV name error is the feed-forward layer named key
wrong = GA["structural"]["per_model"]["RWKV"]["wrong"]
assert len(wrong) == 2 and all(w["tensor"].endswith("feed_forward.key.weight") and w["predicted"] == "softmax_score" for w in wrong)
assert GA["auto"]["per_model"]["RWKV"]["wrong"] == wrong
lab = G["RWKV"]["labelled_by_graph"]
assert sum(k.endswith("time_decay") for k in lab) == 2 and sum(k.endswith("time_first") for k in lab) == 2 and len(lab) == 4
lab = G["Mamba"]["labelled_by_graph"]
assert sum(k.endswith("A_log") for k in lab) == 2 and sum(k.endswith("dt_proj.bias") for k in lab) == 2 and len(lab) == 4
gpt = G["GPT-2"]["mixed_truth"]
assert len(gpt) == 2 and all(x["predicted"] == "softmax_score" for x in gpt)

B = tr["group_b"]
for naming, names_right in [("usual_names", 13), ("uninformative_names", 9)]:
    check(f"Tracer table, small modules, {naming}: scored", 13, B["graph"][f"pooled_{naming}"]["scored"], 0)
    check(f"Tracer table, small modules, {naming}, names only: correct", names_right, B["structural"][f"pooled_{naming}"]["correct"], 0)
    check(f"Tracer table, small modules, {naming}, graph only: correct", 13, B["graph"][f"pooled_{naming}"]["correct"], 0)
    check(f"Tracer table, small modules, {naming}: symbolic trace succeeds", 4, B["graph"][f"pooled_{naming}"]["models_where_symbolic_trace_ok"], 0)
    check(f"Tracer table, small modules, {naming}: forward-pass trace succeeds", 4, B["graph"][f"pooled_{naming}"]["models_where_forward_pass_traced"], 0)
report("Tracer table"); report("Section 6")

Tracer table: 26 of 26 checks pass
Section 6: 2 of 2 checks pass


## Formal checks

`paper/tnnls_operator/lean/OperatorChecks.lean` states the identities the text relies on, in Lean 4
with Mathlib. The cell below prints the build record and verifies that the file on disk is the one
that was built. It does not run Lean.

In [20]:
import hashlib
lean = os.path.join(PAPER, "lean", "OperatorChecks.lean")
record = open(os.path.join(PAPER, "lean", "BUILD-RECORD.txt"), encoding="utf-8").read()
src = open(lean, "rb").read().replace(bytes([13, 10]), bytes([10]))
digest = hashlib.sha256(src).hexdigest()
print(record)
assert digest in record, "the Lean file differs from the one in the build record"
text = src.decode("utf-8")
print("theorems in the file:", len(re.findall(r"^theorem ", text, flags=re.M)))
print("occurrences of the word sorry:", text.count("sorry"))
assert text.count("sorry") == 0

Build record: OperatorChecks.lean
Date: 2026-11-22
File sha256: ed4d52ff18ab4f9aa82f724fad8695ace4b2d529869b6cebc5cda10fac90c4f1
Lean: 4.32.2 (commit f3b06c705e6c85f5314019d5d3baab0fec5b580c)
Mathlib: v4.32.2 (rev 905b95818eb32af7874a58b427f50c1711a5e96c)
Command: lake env lean OperatorChecks.lean, in a project whose lakefile requires Mathlib v4.32.2
Result: exit 0; 0 warnings; 0 errors; 0 occurrences of the word sorry in the file
Theorems: 20, each reported by #print axioms as depending on
  [propext, Classical.choice, Quot.sound] only

base_correct_split, quant_correct_split, regressions_sub_recoveries   (agreement counts)
softmax_row_constant, row_constant_decomposition, key_bit_budget      (scores and the bit budget)
order_kept_of_centered_error, topk_set_kept, topk_set_kept_of_spread,
routing_scale_invariant                                               (top-k routing)
decay_rate, log_decay_rate, steady_state, gain_difference             (state decay)
mem_flat, flat_mono, flat_isC

## Summary

In [21]:
bad = [c for c in CHECKS if not c[3]]
print(f"{len(CHECKS) - len(bad)} of {len(CHECKS)} comparisons with the paper pass")
for where, printed, got, ok in bad:
    print("FAIL", where, "paper", printed, "record", got)
assert not bad

899 of 899 comparisons with the paper pass


## Rerunning the experiments (GPU)

The records above were produced by these scripts. Each needs a GPU and the model weights; they
are not run by this notebook. Set the environment variables shown in each script's docstring.

| Result | Script |
|---|---|
| Table S1 | `experiments/agreement_reference_repeats.py` |
| Table 1 | `experiments/matched_bit_projection_sensitivity.py` |
| Table 2 | `experiments/llamacpp_factorial.py` (needs GGUF files made by llama.cpp) |
| Score-error decomposition | `experiments/score_error_decomposition.py` |
| Table 3 | `experiments/k_wavelength_probe.py` |
| Table 4 | `experiments/k_row_protection_controls.py` |
| Tracer table (supplement, Section S-III) | `experiments/tracer_accuracy.py` (CPU) |
| Table 7 | the routing harness, and `experiments/router_error_margin.py` |
| Section 8 | `benchmarks/validate_mamba_decay.py`, `benchmarks/mamba_decay_grid_census.py` (CPU; set `NB_MODEL` for the second checkpoint) |
| Table 5, Section 5.3 | `experiments/k_quantizer_control.py` (GPU) |
| Table 6, Section 5.4 | `experiments/k_gptq_control.py` (GPU) |
| Section 8, time-constant grids | `benchmarks/mamba_decay_tau_grid.py` (CPU) |
| Section 8, RWKV-4 | `benchmarks/rwkv_decay_grid.py` (CPU) |
| Figure 1 | `paper/tnnls_operator/scripts/make_fig_strata.py` (a four-panel diagram; reads no run record) |